# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# Required environment variables
#   OPENAI_API_KEY="..."
#   CHROMA_OPENAI_API_KEY="..."   # optional; falls back to OPENAI_API_KEY
#   TAVILY_API_KEY="..."          # used by notebook 2


In [4]:
load_dotenv()


True

### VectorDB Instance

In [5]:
chroma_client = chromadb.PersistentClient(path="chromadb")


### Collection

In [6]:
# Pinned model: notebook 2 MUST use the identical model_name when reopening
# the collection, or queries return wrong results (Chroma does not enforce
# embedding-function consistency across sessions).
# api_base supports proxy endpoints (e.g., Udacity Vocareum). Leave OPENAI_BASE_URL
# unset in .env to use the default https://api.openai.com/v1.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL") or None,
    model_name="text-embedding-3-small",
)


In [7]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)


### Add documents

In [8]:
# Guard prevents duplicate-ID errors on re-run.
data_dir = "games"

if collection.count() == 0:
    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            game = json.load(f)

        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
        doc_id = os.path.splitext(file_name)[0]

        # Attach the file-stem ID into metadata so the agent can cite specific records.
        game_with_id = dict(game)
        game_with_id["id"] = doc_id

        collection.add(
            ids=[doc_id],
            documents=[content],
            metadatas=[game_with_id],
        )

print(f"Collection contains {collection.count()} game documents.")


Collection contains 15 game documents.


In [9]:
# === phase1: semantic-search demo ===
# Demonstrates that the vector DB can be queried for semantic search.
demo_queries = [
    "racing simulator on PlayStation",
    "first 3D Mario platformer",
    "Pokemon game for Game Boy",
]

for q in demo_queries:
    results = collection.query(query_texts=[q], n_results=3)
    print(f"\nQuery: {q}")
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    ):
        print(f"  - [{meta['id']}] {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']}) -- distance={dist:.4f}")



Query: racing simulator on PlayStation
  - [001] Gran Turismo (PlayStation 1, 1997) -- distance=0.3553
  - [003] Gran Turismo 5 (PlayStation 3, 2010) -- distance=0.3656
  - [002] Grand Theft Auto: San Andreas (PlayStation 2, 2004) -- distance=0.6360



Query: first 3D Mario platformer
  - [009] Super Mario 64 (Nintendo 64, 1996) -- distance=0.3979
  - [008] Super Mario World (Super Nintendo Entertainment System (SNES), 1990) -- distance=0.4781
  - [010] Super Smash Bros. Melee (GameCube, 2001) -- distance=0.6021



Query: Pokemon game for Game Boy
  - [006] Pokémon Gold and Silver (Game Boy Color, 1999) -- distance=0.4545
  - [007] Pokémon Ruby and Sapphire (Game Boy Advance, 2002) -- distance=0.4828
  - [008] Super Mario World (Super Nintendo Entertainment System (SNES), 1990) -- distance=0.5964
